In [ ]:
from pathlib import Path
import duckdb

db_path = Path("/home/nixos/workspace/TradePilot/data/tradepilot.duckdb")
lake = Path("/home/nixos/workspace/TradePilot/data/lakehouse")

conn = duckdb.connect(str(db_path), read_only=True)

futures_instruments = str(lake / "normalized" / "reference.futures_instruments" / "**" / "*.parquet")
futures_mapping = str(lake / "normalized" / "market.futures_mapping" / "**" / "*.parquet")
futures_daily = str(lake / "normalized" / "market.futures_contract_daily" / "**" / "*.parquet")

In [ ]:
conn.execute("""
SELECT exchange,
       MIN(trade_date) AS min_date,
       MAX(trade_date) AS max_date,
       COUNT(*) AS rows,
       SUM(CASE WHEN is_open THEN 1 ELSE 0 END) AS open_rows
FROM canonical_trading_calendar
WHERE exchange IN ('SHFE', 'DCE', 'CZCE', 'INE')
GROUP BY exchange
ORDER BY exchange
""").fetchdf()

In [ ]:
conn.execute("""
SELECT exchange,
       COUNT(*) AS rows,
       COUNT(DISTINCT contract_code) AS contracts,
       SUM(CASE WHEN multiplier IS NULL THEN 1 ELSE 0 END) AS missing_multiplier,
       SUM(CASE WHEN multiplier <= 0 THEN 1 ELSE 0 END) AS non_positive_multiplier,
       MIN(list_date) AS min_list_date,
       MAX(list_date) AS max_list_date
FROM read_parquet(?)
GROUP BY exchange
ORDER BY exchange
""", [futures_instruments]).fetchdf()

In [ ]:
conn.execute("""
SELECT root_code,
       MIN(trade_date) AS min_date,
       MAX(trade_date) AS max_date,
       COUNT(*) AS rows,
       COUNT(DISTINCT active_contract) AS active_contracts,
       SUM(CASE WHEN active_contract IS NULL THEN 1 ELSE 0 END) AS missing_active_contract
FROM read_parquet(?)
GROUP BY root_code
ORDER BY root_code
""", [futures_mapping]).fetchdf()

In [ ]:
conn.execute("""
SELECT regexp_extract(contract_code, '\\.([A-Z]+)$', 1) AS exchange_suffix,
       MIN(trade_date) AS min_date,
       MAX(trade_date) AS max_date,
       COUNT(*) AS rows,
       COUNT(DISTINCT contract_code) AS contracts,
       SUM(CASE WHEN settle IS NULL THEN 1 ELSE 0 END) AS missing_settle
FROM read_parquet(?)
GROUP BY exchange_suffix
ORDER BY exchange_suffix
""", [futures_daily]).fetchdf()

In [ ]:
conn.execute("""
WITH mapping AS (
  SELECT DISTINCT active_contract
  FROM read_parquet(?)
  WHERE active_contract IS NOT NULL
),
daily AS (
  SELECT DISTINCT contract_code
  FROM read_parquet(?)
)
SELECT COUNT(*) AS missing_contracts
FROM mapping m
LEFT JOIN daily d ON d.contract_code = m.active_contract
WHERE d.contract_code IS NULL
""", [futures_mapping, futures_daily]).fetchdf()

In [ ]:
conn.execute("""
WITH mapping AS (
  SELECT root_code, trade_date, active_contract
  FROM read_parquet(?)
),
daily AS (
  SELECT contract_code, trade_date
  FROM read_parquet(?)
)
SELECT COUNT(*) AS missing_same_day_daily_rows
FROM mapping m
LEFT JOIN daily d
  ON d.contract_code = m.active_contract
 AND d.trade_date = m.trade_date
WHERE d.contract_code IS NULL
""", [futures_mapping, futures_daily]).fetchdf()

In [ ]:
conn.execute("""
WITH mapping_dup AS (
  SELECT 'market.futures_mapping' AS dataset, COUNT(*) AS duplicate_rows
  FROM (
    SELECT root_code, trade_date, COUNT(*) AS n
    FROM read_parquet(?)
    GROUP BY root_code, trade_date
    HAVING COUNT(*) > 1
  )
),
daily_dup AS (
  SELECT 'market.futures_contract_daily' AS dataset, COUNT(*) AS duplicate_rows
  FROM (
    SELECT contract_code, trade_date, COUNT(*) AS n
    FROM read_parquet(?)
    GROUP BY contract_code, trade_date
    HAVING COUNT(*) > 1
  )
),
instrument_dup AS (
  SELECT 'reference.futures_instruments' AS dataset, COUNT(*) AS duplicate_rows
  FROM (
    SELECT contract_code, COUNT(*) AS n
    FROM read_parquet(?)
    GROUP BY contract_code
    HAVING COUNT(*) > 1
  )
)
SELECT * FROM mapping_dup
UNION ALL SELECT * FROM daily_dup
UNION ALL SELECT * FROM instrument_dup
""", [futures_mapping, futures_daily, futures_instruments]).fetchdf()

In [ ]:
conn.execute("""
SELECT
  SUM(CASE WHEN open < 0 OR high < 0 OR low < 0 OR close < 0 OR settle < 0 THEN 1 ELSE 0 END) AS negative_price_rows,
  SUM(CASE WHEN volume < 0 THEN 1 ELSE 0 END) AS negative_volume_rows,
  SUM(CASE WHEN oi < 0 THEN 1 ELSE 0 END) AS negative_oi_rows,
  SUM(CASE WHEN high < GREATEST(open, low, close) THEN 1 ELSE 0 END) AS bad_high_rows,
  SUM(CASE WHEN low > LEAST(open, high, close) THEN 1 ELSE 0 END) AS bad_low_rows
FROM read_parquet(?)
""", [futures_daily]).fetchdf()

In [ ]:
conn.execute("""
SELECT run_id,
       dataset_name,
       status,
       records_discovered,
       records_inserted,
       records_updated,
       error_message,
       started_at,
       finished_at
FROM etl_ingestion_runs
WHERE dataset_name IN (
  'reference.trading_calendar',
  'reference.futures_instruments',
  'market.futures_mapping',
  'market.futures_contract_daily'
)
ORDER BY run_id DESC
LIMIT 20
""").fetchdf()

In [ ]:
conn.execute("""
SELECT contract_code,
       trade_date,
       open,
       high,
       low,
       close,
       settle,
       volume,
       amount,
       oi
FROM read_parquet(?)
WHERE high < GREATEST(open, low, close)
ORDER BY trade_date, contract_code
""", [futures_daily]).fetchdf()

In [ ]:
conn.execute("""
WITH bad AS (
  SELECT contract_code, trade_date, open, high, low, close
  FROM read_parquet(?)
  WHERE high < GREATEST(open, low, close)
),
mapping AS (
  SELECT root_code, trade_date, active_contract
  FROM read_parquet(?)
)
SELECT b.*,
       m.root_code
FROM bad b
LEFT JOIN mapping m
  ON m.active_contract = b.contract_code
 AND m.trade_date = b.trade_date
ORDER BY b.trade_date, b.contract_code
""", [futures_daily, futures_mapping]).fetchdf()